# prep.tableS4

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from daforfer import DaforferDB
from skbio.diversity.alpha import chao1
import matplotlib.pyplot as plt
import networkx as nx
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

/home/bcz/miniconda3/envs/miripvir25/lib/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


┌─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  name   │                                                        description                                                        │
│ varchar │                                                          varchar                                                          │
├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ TableS1 │ Table S1: Library sites and context                                                                                       │
│ TableS2 │ This table summarizes most of the information of our detected OTUs, including host_range, site_range, habitat_range, etc. │
│ TableS3 │ Site-level diversity and number of cooccurring virus-bacteria                                                             │
│ TableS4 │ Habitat-level diversity and number o

## PAB diversity

In [2]:
metadata = db.conn.sql('SELECT * FROM D_sites').df()
bacteria_hits = db.conn.sql('SELECT * FROM D_PABHits').df()
bacteria_hits = pd.merge(metadata, bacteria_hits, on='library', how='left')
bacteria_hits

,site,library,habitat,n_extracts,host_taxon,taxid,scientific_name,is_pab,pab_type
0,C1,PV534,Crop,3,Diplotaxis erucoides,NaN,NaN,NaN,NaN
1,C1,PV535,Crop,17,Brassica oleracea,1563157.0,Pseudomonas endophytica,True,pab_unknown
2,C1,PV535,Crop,17,Brassica oleracea,1270.0,Micrococcus luteus,True,pab_unknown
3,C1,PV538,Crop,8,Brassica oleracea,NaN,NaN,NaN,NaN
4,C1,PV540,Crop,1,Picris echioides,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
626,Z2,PV527,Crop,4,Convolvulus arvensis,1770058.0,Devosia elaeis,True,pab_unknown
627,Z2,PV529,Crop,1,Picris echioides,47880.0,Pseudomonas fulva,True,pab_unknown
628,Z2,PV529,Crop,1,Picris echioides,1220495.0,Pseudomonas punonensis,True,pab_unknown
629,Z2,PV529,Crop,1,Picris echioides,289370.0,Pseudomonas argentinensis,True,pab_unknown


We simply create a list of item counts in one of the columns. 

In [3]:
pab_alpha_diversity = bacteria_hits.value_counts(
    ['habitat', 'scientific_name']
    ).reset_index().groupby(
        ['habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
pab_alpha_diversity

,habitat,hits
0,Crop,"[7, 5, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ..."
1,Edge,"[15, 13, 10, 10, 6, 6, 6, 5, 4, 4, 4, 4, 4, 4,..."
2,Oak,"[9, 4, 3, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, ..."
3,Wasteland,"[11, 10, 8, 6, 6, 5, 5, 4, 4, 3, 3, 2, 2, 2, 2..."


In [4]:
pab_alpha_diversity['species_richness'] = pab_alpha_diversity['hits'].apply(lambda x: len(x))
pab_alpha_diversity['chao1'] = pab_alpha_diversity['hits'].apply(chao1)
pab_alpha_diversity = pab_alpha_diversity.sort_values(by=['habitat'])
pab_alpha_diversity['disturbed'] = pab_alpha_diversity['habitat'].apply(
    lambda x: {"Crop": "disturbed", "Wasteland": "non-disturbed", "Edge": "disturbed", "Oak": "non-disturbed"}[x]
)
pab_alpha_diversity = pab_alpha_diversity.drop(columns=['hits'])[['habitat', 'disturbed', 'species_richness', 'chao1']]
pab_alpha_diversity

,habitat,disturbed,species_richness,chao1
0,Crop,disturbed,52,103.230769
1,Edge,disturbed,77,147.714286
2,Oak,non-disturbed,26,45.428571
3,Wasteland,non-disturbed,61,98.187500


## Virus diversity

In [5]:
# virus_hits = pd.read_csv("output/hits.virus.csv", sep=";")
virus_hits = db.conn.sql('SELECT * FROM D_virusHits').df()
virus_hits = pd.merge(metadata, virus_hits, on='library', how='left')
virus_alpha_diversity = virus_hits.value_counts(
    ['habitat', 'scientific_name']
    ).reset_index().groupby(
        ['habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
virus_alpha_diversity['species_richness'] = virus_alpha_diversity['hits'].apply(lambda x: len(x))
virus_alpha_diversity['chao1'] = virus_alpha_diversity['hits'].apply(chao1)
virus_alpha_diversity = virus_alpha_diversity.drop(columns=['hits'])[['habitat',  'species_richness', 'chao1']]
virus_alpha_diversity


,habitat,species_richness,chao1
0,Crop,64,89.666667
1,Edge,113,165.555556
2,Oak,39,49.500000
3,Wasteland,59,129.300000


## Plant diversity

In [6]:
plant_hits = pd.read_csv("input/hits.plant.csv", sep=';', header=None, names=['Code', 'Species name']).dropna()
plant_hits['Collection_code'] = plant_hits['Code'].apply(lambda x: x.split('.')[0])
plant_hits['position'] = plant_hits['Code'].apply(lambda x: int(x.split('.')[1]))
plant_hits = pd.merge(
    pd.read_csv("input/mcleish24.TableS1.csv", sep=';'), plant_hits, on='Collection_code',
    how='left'
)
plant_hits = plant_hits[['Site_code', 'Species name']].drop_duplicates(['Site_code', 'Species name']).rename(columns={'Site_code': 'site'})
plant_hits = pd.merge(
    metadata[['site', 'habitat']].drop_duplicates(['site', 'habitat'], keep='first'), 
    plant_hits, on='site', how='left'
).dropna(subset='Species name')
plant_alpha_diversity = plant_hits.value_counts(
    ['habitat', 'Species name']
    ).reset_index().groupby(
        ['habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
plant_alpha_diversity['species_richness'] = plant_alpha_diversity['hits'].apply(lambda x: len(x))
plant_alpha_diversity['chao1'] = plant_alpha_diversity['hits'].apply(chao1)
plant_alpha_diversity = plant_alpha_diversity.drop(columns=['hits'])[['habitat',  'species_richness', 'chao1']]
plant_alpha_diversity

,habitat,species_richness,chao1
0,Crop,73,108.150000
1,Edge,94,220.000000
2,Oak,100,163.103448
3,Wasteland,126,230.500000


## Host diversity

In [7]:
host_alpha_diversity = metadata.value_counts(
    ['habitat', 'host_taxon']
).reset_index().groupby(
    ['habitat']
)['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
host_alpha_diversity['species_richness'] = host_alpha_diversity['hits'].apply(lambda x: len(x))
host_alpha_diversity['chao1'] = host_alpha_diversity['hits'].apply(chao1)
host_alpha_diversity = host_alpha_diversity.drop(columns=['hits'])[['habitat',  'species_richness', 'chao1']]
host_alpha_diversity

,habitat,species_richness,chao1
0,Crop,28,58.60
1,Edge,52,71.25
2,Oak,51,97.75
3,Wasteland,52,149.50


## Merge

In [8]:
# bacteria_alpha_diversity = db.conn.query('SELECT * FROM D_PABAlphaDiv').df()
alpha_diversity = pd.merge(
    pab_alpha_diversity[['habitat', 'disturbed', 'species_richness', 'chao1']], 
    virus_alpha_diversity[['habitat', 'species_richness', 'chao1']],
    on='habitat', suffixes=['_bact', '_vir']
)

alpha_diversity = pd.merge(
    alpha_diversity,
    plant_alpha_diversity[['habitat', 'species_richness', 'chao1']].rename(
        columns={
            'species_richness': 'species_richness_plant',
            'chao1': 'chao1_plant',
        }
    ),
    on='habitat'
)

alpha_diversity = pd.merge(
    alpha_diversity,
    host_alpha_diversity[['habitat', 'species_richness', 'chao1']].rename(
        columns={
            'species_richness': 'species_richness_host',
            'chao1': 'chao1_host',
        }
    ),
    on='habitat'
)


# alpha_diversity.to_csv("output/diversity.all.csv", sep=";", index=False)
# db.save_dataframe(
#     df=alpha_diversity, table_name="D_ADAllOrganismsSite",
#     description="Alpha diversity per site for virus, plant, host and bacteria at site level"
# )

alpha_diversity

,habitat,disturbed,species_richness_bact,chao1_bact,species_richness_vir,chao1_vir,species_richness_plant,chao1_plant,species_richness_host,chao1_host
0,Crop,disturbed,52,103.230769,64,89.666667,73,108.150000,28,58.60
1,Edge,disturbed,77,147.714286,113,165.555556,94,220.000000,52,71.25
2,Oak,non-disturbed,26,45.428571,39,49.500000,100,163.103448,51,97.75
3,Wasteland,non-disturbed,61,98.187500,59,129.300000,126,230.500000,52,149.50


## Cooccurrences by site

In [9]:
cooccurrence_network = nx.read_graphml("output/network.coocurrence.virusbact-bylibrary.trans.graphml")
cooccurrence_network.number_of_edges()

57

We need to use original hit tables, as we will use the OTUs scientific names to make subgraphs on the cooccurrence network.

In [10]:
hits_by_habitat = pd.concat([
    bacteria_hits[['habitat', 'scientific_name']],
    virus_hits[['habitat', 'scientific_name']],
]).drop_duplicates(['habitat', 'scientific_name'], keep='first')
hits_by_habitat

,habitat,scientific_name
0,Crop,NaN
1,Crop,Pseudomonas endophytica
2,Crop,Micrococcus luteus
5,Crop,Sphingomonas sp. Leaf20
6,Crop,Rhodococcoides fascians
...,...,...
1631,Crop,Rehmannia mosaic virus
1632,Crop,Soybean mild mottle pararetrovirus
1635,Crop,Tomato mosaic virus
1638,Crop,Pepper cryptic virus 2




We look for subgraphs of the cooccurrence graph within each of the sites

In [11]:
cooccurrences_by_habitat = []
for habitat in hits_by_habitat.habitat.unique():
    habitat_cooccurrence_subnetwork = cooccurrence_network.subgraph(
        hits_by_habitat.query('habitat == "{:s}"'.format(habitat))['scientific_name'].to_list()
    )
    cooccurrences_by_habitat.append({
        "habitat": habitat,
        "total_cooccurrences": habitat_cooccurrence_subnetwork.number_of_edges()
    })
cooccurrences_by_habitat = pd.DataFrame.from_records(cooccurrences_by_habitat)
cooccurrences_by_habitat = pd.merge(
    cooccurrences_by_habitat, # type: ignore
    hits_by_habitat.drop_duplicates(['habitat'], keep='first')[['habitat']]
)
cooccurrences_by_habitat

,habitat,total_cooccurrences
0,Crop,40
1,Wasteland,31
2,Edge,50
3,Oak,20


In [12]:
tableS4 = pd.merge(alpha_diversity, cooccurrences_by_habitat, on=['habitat'])
tableS4

,habitat,disturbed,species_richness_bact,chao1_bact,species_richness_vir,chao1_vir,species_richness_plant,chao1_plant,species_richness_host,chao1_host,total_cooccurrences
0,Crop,disturbed,52,103.230769,64,89.666667,73,108.150000,28,58.60,40
1,Edge,disturbed,77,147.714286,113,165.555556,94,220.000000,52,71.25,50
2,Oak,non-disturbed,26,45.428571,39,49.500000,100,163.103448,51,97.75,20
3,Wasteland,non-disturbed,61,98.187500,59,129.300000,126,230.500000,52,149.50,31


## Save

In [13]:
db.save_dataframe(
    tableS4, "D_Habitat_level_div",
    description="Habitat-level diversity and number of cooccurring virus-bacteria"
)

Saved D_Habitat_level_div to db.2025-10-27


In [14]:
si.save_dataframe(
    tableS4, "TableS4",
    description="Habitat-level diversity and number of cooccurring virus-bacteria"
)

Saved TableS4 to si.2025-10-27


In [15]:
tableS4.to_csv("output/TableS4.csv", sep=";")

In [16]:
si.conn.close()
db.conn.close()